# Shift Scheduler Optimizer - Walkthrough Explicativo 🚚🕒

Bem-vindo ao guia passo a passo do **Shift Scheduler Optimizer**! Este notebook foi estruturado para demonstrar como aplicar conceitos de **Pesquisa Operacional (PO)** e **Engenharia de Produção** para resolver um dos problemas mais complexos da gestão logística de alta performance: a escala otimizada de turnos operacionais 24/7.

---

## Passo 1: Importação de Bibliotecas e Preparação dos Dados

Primeiro, vamos importar os pacotes necessários: `pandas` para a modelagem dos dados, `pulp` para resolver a programação linear de inteiros mistos (MILP) e `matplotlib` para o output visual.

In [ ]:
import os
import pandas as pd
import pulp
import matplotlib.pyplot as plt
import numpy as np

print("✅ Todas as bibliotecas foram importadas com sucesso!")

## Passo 2: Carregando a Visão do Planejamento (CSV)

Vamos ler a matriz de perfil de custo dos colaboradores e a demanda mínima prevista para os postos de trabalho ao longo da semana.

In [ ]:
# Lendo os dados gerados previamente
df_demand = pd.read_csv("../data/demand_shifts.csv")
df_workers = pd.read_csv("../data/workers_profile.csv")

print("--- Perfil dos Trabalhadores --- ")
display(df_workers.head(3))
print("\n--- Exemplo de Demanda por Turno ---")
display(df_demand.head(3))

## Passo 3: Formulação Matemática via PuLP

Abaixo, instanciamos o modelo de **minimização de custo** e declaramos a nossa variável de decisão binária:
$$x_{i,d,t} \in \{0, 1\}$$
Onde $1$ significa que o operador foi alocado, e $0$ indica folga ou descanso.

In [ ]:
workers = df_workers["worker_id"].tolist()
days = sorted(df_demand["day_of_week"].unique())
shifts = ["A", "B", "C"]

worker_costs = pd.Series(df_workers.hourly_rate.values, index=df_workers.worker_id).to_dict()
worker_night_pct = pd.Series(df_workers.night_premium_pct.values, index=df_workers.worker_id).to_dict()
demand_dict = df_demand.set_index(["day_of_week", "shift"])["min_workers"].to_dict()

# Inicializar problema matemático
model = pulp.LpProblem("Notebook_Shift_Optimizer", pulp.LpMinimize)

# Dicionário de variáveis de decisão binárias
x = pulp.LpVariable.dicts("assign", ((i, d, t) for i in workers for d in days for t in shifts), cat=pulp.LpBinary)
print(f"Estrutura construída: {len(x)} variáveis binárias geradas no modelo.")

## Passo 4: Adicionando Restrições Operacionais e Trabalhistas (CLT)

Agora aplicamos as equações limites para garantir que o modelo gere uma escala factível e segura para o fator humano, mitigando passivos trabalhistas.

In [ ]:
# Função Objetivo: Custos Diretos + Adicional Noturno integrado no Turno A
total_cost = []
for i in workers:
    for d in days:
        for t in shifts:
            cost = worker_costs[i] * 8
            if t == "A":
                cost *= (1 + worker_night_pct[i])
            total_cost.append(x[i, d, t] * cost)
model += pulp.lpSum(total_cost)

# Restrição 1: Garantir atendimento da demanda logística mínima em cada turno
for d in days:
    for t in shifts:
        model += pulp.lpSum(x[i, d, t] for i in workers) >= demand_dict[(d, t)]

# Restrição 2: Ergonomia - Limite de 1 turno por dia por colaborador
for i in workers:
    for d in days:
        model += pulp.lpSum(x[i, d, t] for t in shifts) <= 1

# Restrição 3: Descanso Interjornada CLT (Proibido Turno C seguido de Turno A no dia posterior)
for i in workers:
    for d in days:
        next_day = (d + 1) % 7
        model += x[i, d, "C"] + x[i, next_day, "A"] <= 1

# Restrição 4: Jornada máxima contratual (44 horas semanais)
for i in workers:
    model += pulp.lpSum(x[i, d, t] * 8 for d in days for t in shifts) <= 44

print("⚙️ Todas as restrições matemáticas injetadas com sucesso no solver!")

## Passo 5: Resolução e Plotagem do Gráfico de Gantt

Vamos rodar o algoritmo de otimização e transformar os arrays booleanos de saída em uma visualização de Gantt elegante para o acompanhamento dos supervisores da operação.

In [ ]:
status = model.solve(pulp.PULP_CBC_CMD(msg=False))
print(f"Status do Solver: {pulp.LpStatus[status]}")

if pulp.LpStatus[status] == "Optimal":
    # Coletando resultados para visualização imediata no notebook
    output = []
    for d in days:
        for t in shifts:
            for i in workers:
                if pulp.value(x[i, d, t]) == 1:
                    output.append({"Worker": i, "Day": d, "Shift": t})
    
    df_res = pd.DataFrame(output)
    
    # Geração rápida do gráfico inline
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c"}
    
    for _, row in df_res.iterrows():
        start_hour = (row["Day"] * 24) + (0 if row["Shift"] == "A" else 8 if row["Shift"] == "B" else 16)
        y_pos = workers.index(row["Worker"])
        ax.broken_barh([(start_hour, 8)], (y_pos - 0.3, 0.6), facecolors=colors[row["Shift"]])
        
    ax.set_yticks(range(len(workers)))
    ax.set_yticklabels(workers)
    ax.set_xticks(np.arange(0, 168, 24))
    ax.set_xticklabels(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
    ax.set_title("Escala Visualizada Dinamicamente no Jupyter")
    plt.grid(axis='x', linestyle='--')
    plt.show()